In [11]:
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

# ==============================
# 1. Load Data
# ==============================
df = pd.read_excel(r"C:\Users\ahmed3farag\Downloads\RAW DATA-ev.xlsx")

# تنظيف الأعمدة
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# ==============================
# 2. Detect Columns
# ==============================
year_col = [c for c in df.columns if 'year' in c][0]
country_col = [c for c in df.columns if 'country' in c][0]

sales_col = None
for c in df.columns:
    if 'sales' in c:
        sales_col = c
        break

if sales_col is None:
    sales_col = df.columns[-1]

# ==============================
# 3. Clean Data
# ==============================
df[year_col] = pd.to_numeric(df[year_col], errors='coerce')
df = df.dropna(subset=[year_col])

# ==============================
# 4. Create App
# ==============================
app = dash.Dash(__name__)

app.layout = html.Div([

    html.H1("EV Interactive Dashboard", style={'textAlign': 'center'}),

    # ==============================
    # Filters
    # ==============================
    dcc.Dropdown(
        id='country-filter',
        options=[{'label': c, 'value': c} for c in df[country_col].dropna().unique()],
        multi=True,
        placeholder="Select Country"
    ),

    dcc.RangeSlider(
        id='year-filter',
        min=int(df[year_col].min()),
        max=int(df[year_col].max()),
        value=[int(df[year_col].min()), int(df[year_col].max())],
        marks={i: str(i) for i in range(int(df[year_col].min()), int(df[year_col].max())+1, 2)}
    ),

    # ==============================
    # KPIs
    # ==============================
    html.Div([
        html.Div(id='kpi-total', style={'padding': '20px', 'background': '#eee', 'borderRadius': '10px', 'flex': '1'}),
        html.Div(id='kpi-avg', style={'padding': '20px', 'background': '#eee', 'borderRadius': '10px', 'flex': '1'}),
        html.Div(id='kpi-count', style={'padding': '20px', 'background': '#eee', 'borderRadius': '10px', 'flex': '1'}),
    ], style={'display': 'flex', 'gap': '20px', 'marginTop': '20px'}),

    # ==============================
    # Chart
    # ==============================
    dcc.Graph(id='main-chart')

])

# ==============================
# 5. Callback
# ==============================
@app.callback(
    [
        Output('main-chart', 'figure'),
        Output('kpi-total', 'children'),
        Output('kpi-avg', 'children'),
        Output('kpi-count', 'children')
    ],
    [
        Input('country-filter', 'value'),
        Input('year-filter', 'value')
    ]
)
def update_dashboard(countries, years):

    dff = df.copy()

    # Filter by country
    if countries:
        dff = dff[dff[country_col].isin(countries)]

    # Filter by year
    dff = dff[
        (dff[year_col] >= years[0]) &
        (dff[year_col] <= years[1])
    ]

    # KPIs
    total = dff[sales_col].sum()
    avg = dff[sales_col].mean()
    count = dff.shape[0]

    # Chart
    fig = px.line(
        dff,
        x=year_col,
        y=sales_col,
        color=country_col,
        title="EV Sales Over Time"
    )

    return (
        fig,
        f"Total Sales: {int(total):,}",
        f"Average: {round(avg,2)}",
        f"Records: {count}"
    )

# ==============================
# 6. Run App
# ==============================
if __name__ == '__main__':
    app.run(debug=True)